# Notebook 47 - UltraTrack KLT sequential refresh variants

Notebook 46 showed that Kalman changes the final MATLAB output, but does not explain the raw KLT mismatch against `fas_x_original/fas_y_original`. Notebook 45 showed that local one-step KLT/affine estimates are close.

This notebook tests the next hypothesis:

> Is the sequential drift caused by persistent point-tracker state, or by compounding small affine errors over many frames?

It compares four fascicle-only variants using the same MATLAB oracle masks:

1. **persistent_cumulative**: MATLAB-like persistent points; apply each affine to previous Python geometry.
2. **persistent_one_step**: MATLAB-like persistent points; apply each affine to MATLAB geometry at `f-1`.
3. **redetect_cumulative**: re-detect points on every transition; apply each affine to previous Python geometry.
4. **redetect_one_step**: re-detect points on every transition; apply each affine to MATLAB geometry at `f-1`.

If re-detect cumulative still drifts while re-detect one-step is good, then point refresh alone is not enough; the issue is cumulative affine integration.

In [1]:
from pathlib import Path
import json
import os
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import cv2
import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row

OUT = ROOT / "results" / "notebook47_ultratrack_klt_sequential_refresh_variants"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
VIDEO_PATH = ROOT / "data" / "raw" / "Test2.mp4"
NB44_METRICS = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "klt_oracle_mask_full_metrics.csv"
NB45_METRICS = ROOT / "results" / "notebook45_ultratrack_klt_one_step_affine_diagnostics" / "klt_one_step_affine_metrics.csv"

for path in [MATLAB_EXPORT, VIDEO_PATH, NB44_METRICS, NB45_METRICS]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output folder:", OUT)
print("Video:", VIDEO_PATH)

Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants
Video: /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4


## Load MATLAB raw KLT target

The target remains MATLAB pre-Kalman KLT geometry: `Region.Fascicle.fas_x_original` / `fas_y_original`. Endpoint order is converted from MATLAB `[deep; superficial]` to Python `[x_sup, y_sup, x_deep, y_deep]`.

In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))
parms = mat_root["parms"]

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height
block_size_matlab = np.asarray(mat_root["BlockSize"], dtype=int).reshape(-1)
contract_win_size = (int(block_size_matlab[1]), int(block_size_matlab[0]))

cap = cv2.VideoCapture(str(VIDEO_PATH))
video_info = {
    "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "fps": float(cap.get(cv2.CAP_PROP_FPS)),
}
cap.release()
if (video_info["height"], video_info["width"]) != (height, width):
    raise ValueError(f"Video shape mismatch: {video_info} vs {(height, width)}")

print({
    "frames": mat_n,
    "shape": (height, width),
    "mm_per_px": mm_per_px,
    "matlab_block_size_height_width": block_size_matlab.tolist(),
    "opencv_win_size_width_height": contract_win_size,
})

{'frames': 2666, 'shape': (562, 706), 'mm_per_px': 0.09021352313167261, 'matlab_block_size_height_width': [21, 71], 'opencv_win_size_width_height': (71, 21)}


In [3]:
def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


def angle_delta_deg(new_angle: np.ndarray, old_angle: np.ndarray) -> np.ndarray:
    return (np.asarray(new_angle) - np.asarray(old_angle) + 90.0) % 180.0 - 90.0


mat_fascicle = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_angle = normalized_angles_deg(mat_fascicle)
mat_length_px = line_lengths_batch(mat_fascicle)

print("MATLAB raw fascicle frame 0:", mat_fascicle[0])
print("MATLAB raw fascicle frame 1:", mat_fascicle[1])

MATLAB raw fascicle frame 0: [733.          54.41875512 -27.         309.01343161]
MATLAB raw fascicle frame 1: [732.97476427  54.42288198 -27.01335866 309.01276553]


## Oracle KLT masks and OpenCV helpers

This uses the same MATLAB-oracle mask construction as notebooks 44 and 45. The only thing varied in this notebook is how the point set and geometry state are refreshed between frames.

In [4]:
super_cut = np.asarray(parms["apo"]["super"]["cut"], dtype=float).reshape(-1)
deep_cut = np.asarray(parms["apo"]["deep"]["cut"], dtype=float).reshape(-1)


def poly_mask_1b(x_values, y_values, shape=(height, width)) -> np.ndarray:
    x = np.asarray(x_values, dtype=float).reshape(-1) - 1.0
    y = np.asarray(y_values, dtype=float).reshape(-1) - 1.0
    points = np.rint(np.column_stack([x, y])).astype(np.int32)
    mask = np.zeros(shape, dtype=np.uint8)
    cv2.fillPoly(mask, [points], 1)
    return mask.astype(bool)


def tracking_masks(frame_index: int) -> dict[str, np.ndarray]:
    entry = geofeatures[frame_index]

    line_mask = np.zeros((height, width), dtype=bool)
    xs = np.asarray(entry["x"], dtype=float)
    ys = np.asarray(entry["y"], dtype=float)
    for (x1, x2), (y1, y2) in zip(xs, ys):
        px = [x1, x1, x2, x2]
        py = [y1 - 5.0, y1 + 5.0, y2 + 5.0, y2 - 5.0]
        if np.all(np.isfinite(px)) and np.all(np.isfinite(py)):
            line_mask |= poly_mask_1b(px, py)

    super_pos = np.asarray(entry["super_pos"], dtype=float).reshape(-1)
    deep_pos = np.asarray(entry["deep_pos"], dtype=float).reshape(-1)
    thickness = deep_pos - super_pos
    r = 0.1
    roix = [1.0, 1.0, float(width), float(width), 1.0]
    roiy_fcor = np.rint([
        super_pos[0] + thickness[0] * r,
        deep_pos[0] - thickness[0] * r,
        deep_pos[1] - thickness[1] * r,
        super_pos[1] + thickness[1] * r,
        super_pos[0] + thickness[0] * r,
    ])
    fcor_mask = poly_mask_1b(roix, roiy_fcor)

    return {
        "line_mask": line_mask,
        "fcor_mask": fcor_mask,
        "fascicle_mask": line_mask & fcor_mask,
    }


def masked_image(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    out = np.asarray(gray).copy()
    out[~mask] = 0
    return out


def detect_min_eigen_like(gray: np.ndarray, mask: np.ndarray, max_corners: int = 300) -> np.ndarray:
    img = masked_image(gray, mask)
    points = cv2.goodFeaturesToTrack(
        img,
        maxCorners=max_corners,
        qualityLevel=0.005,
        minDistance=1,
        blockSize=11,
        useHarrisDetector=False,
    )
    if points is None:
        return np.empty((0, 1, 2), dtype=np.float32)
    return points.astype(np.float32)


def filter_points_by_mask(points: np.ndarray, mask: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    if len(points) == 0:
        return points.reshape(0, 1, 2)
    xy = np.rint(points).astype(int)
    xy[:, 0] = np.clip(xy[:, 0], 0, width - 1)
    xy[:, 1] = np.clip(xy[:, 1], 0, height - 1)
    keep = mask[xy[:, 1], xy[:, 0]]
    return points[keep].reshape(-1, 1, 2).astype(np.float32)


def detect_fascicle_points(gray: np.ndarray, frame_index: int) -> np.ndarray:
    m = tracking_masks(frame_index)
    points = detect_min_eigen_like(gray, m["fascicle_mask"], max_corners=300)
    return filter_points_by_mask(points, m["fcor_mask"])


def track_points(prev_gray: np.ndarray, gray: np.ndarray, points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    points = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
    if len(points) == 0:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    new_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        gray,
        points,
        None,
        winSize=contract_win_size,
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 50, 0.01),
    )
    if new_points is None or status is None:
        return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
    keep = status.reshape(-1).astype(bool)
    return points.reshape(-1, 2)[keep], new_points.reshape(-1, 2)[keep]


def estimate_affine_matlab_coords(old_points_0b: np.ndarray, new_points_0b: np.ndarray) -> tuple[np.ndarray | None, int]:
    old_points_0b = np.asarray(old_points_0b, dtype=np.float32).reshape(-1, 2)
    new_points_0b = np.asarray(new_points_0b, dtype=np.float32).reshape(-1, 2)
    if len(old_points_0b) < 3:
        return None, 0
    affine, inliers = cv2.estimateAffine2D(
        old_points_0b + 1.0,
        new_points_0b + 1.0,
        method=cv2.RANSAC,
        ransacReprojThreshold=50.0,
        maxIters=2000,
        confidence=0.99,
        refineIters=10,
    )
    if affine is None:
        return None, 0
    n_inliers = int(np.asarray(inliers).sum()) if inliers is not None else len(old_points_0b)
    return affine.astype(np.float32), n_inliers


def apply_affine_1b(segment_1b: np.ndarray, affine: np.ndarray) -> np.ndarray:
    points = np.asarray(segment_1b, dtype=np.float32).reshape(1, -1, 2)
    return cv2.transform(points, affine.astype(np.float32))[0].reshape(-1).astype(float)

## Run sequential variants

The notebook estimates two affine streams:

- **persistent**: points are tracked forward and refreshed only by MATLAB's `<100` rule.
- **redetect**: points are re-detected in the previous frame for every transition.

Each affine stream is applied in two ways:

- **cumulative**: affine is applied to the previous Python estimate.
- **one-step**: affine is applied to MATLAB geometry at `f-1`.

In [5]:
def read_next_gray(cap: cv2.VideoCapture) -> np.ndarray:
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError("Could not read next video frame")
    if frame.ndim == 3:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return frame.copy()


def run_refresh_variants() -> dict:
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    prev_gray = read_next_gray(cap)

    persistent_points = detect_fascicle_points(prev_gray, 0)

    outputs = {
        "persistent_cumulative": np.full((mat_n, 4), np.nan, dtype=np.float32),
        "persistent_one_step": np.full((mat_n, 4), np.nan, dtype=np.float32),
        "redetect_cumulative": np.full((mat_n, 4), np.nan, dtype=np.float32),
        "redetect_one_step": np.full((mat_n, 4), np.nan, dtype=np.float32),
    }
    for arr in outputs.values():
        arr[0] = mat_fascicle[0]

    p_affine = np.full((mat_n, 2, 3), np.nan, dtype=np.float32)
    r_affine = np.full((mat_n, 2, 3), np.nan, dtype=np.float32)
    counts = []

    start = time.time()
    for frame in range(1, mat_n):
        gray = read_next_gray(cap)

        old_p, new_p = track_points(prev_gray, gray, persistent_points)
        affine_p, inliers_p = estimate_affine_matlab_coords(old_p, new_p)
        if affine_p is not None:
            outputs["persistent_cumulative"][frame] = apply_affine_1b(outputs["persistent_cumulative"][frame - 1], affine_p)
            outputs["persistent_one_step"][frame] = apply_affine_1b(mat_fascicle[frame - 1], affine_p)
            p_affine[frame] = affine_p

        # MATLAB-like setPoints update for the persistent stream.
        m_current = tracking_masks(frame)
        persistent_points = np.asarray(new_p, dtype=np.float32).reshape(-1, 1, 2)
        reinit_persistent = False
        if len(persistent_points) < 100:
            persistent_points = detect_min_eigen_like(gray, m_current["fascicle_mask"], max_corners=300)
            reinit_persistent = True
        persistent_points = filter_points_by_mask(persistent_points, m_current["fcor_mask"])

        redetect_points = detect_fascicle_points(prev_gray, frame - 1)
        old_r, new_r = track_points(prev_gray, gray, redetect_points)
        affine_r, inliers_r = estimate_affine_matlab_coords(old_r, new_r)
        if affine_r is not None:
            outputs["redetect_cumulative"][frame] = apply_affine_1b(outputs["redetect_cumulative"][frame - 1], affine_r)
            outputs["redetect_one_step"][frame] = apply_affine_1b(mat_fascicle[frame - 1], affine_r)
            r_affine[frame] = affine_r

        counts.append({
            "frame": frame,
            "persistent_tracked_points": int(len(new_p)),
            "persistent_points_after_refresh": int(len(persistent_points)),
            "redetect_points": int(len(redetect_points)),
            "persistent_inliers": int(inliers_p),
            "redetect_inliers": int(inliers_r),
            "persistent_affine_ok": affine_p is not None,
            "redetect_affine_ok": affine_r is not None,
            "persistent_reinitialized": reinit_persistent,
        })

        prev_gray = gray

        if frame % 500 == 0 or frame == mat_n - 1:
            print(
                f"processed {frame + 1}/{mat_n} | "
                f"persistent_pts={len(persistent_points)} redetect_pts={len(redetect_points)} "
                f"elapsed={time.time() - start:.1f}s"
            )

    cap.release()
    return {
        "outputs": outputs,
        "persistent_affine": p_affine,
        "redetect_affine": r_affine,
        "counts": pd.DataFrame(counts),
        "elapsed_s": time.time() - start,
    }


variant_result = run_refresh_variants()

out_npz = OUT / "klt_refresh_variant_arrays.npz"
np.savez(
    out_npz,
    frame=np.arange(mat_n, dtype=np.int32),
    persistent_cumulative=variant_result["outputs"]["persistent_cumulative"],
    persistent_one_step=variant_result["outputs"]["persistent_one_step"],
    redetect_cumulative=variant_result["outputs"]["redetect_cumulative"],
    redetect_one_step=variant_result["outputs"]["redetect_one_step"],
    persistent_affine=variant_result["persistent_affine"],
    redetect_affine=variant_result["redetect_affine"],
    mm_per_px=np.asarray(mm_per_px, dtype=np.float32),
    win_size=np.asarray(contract_win_size, dtype=np.int32),
)
counts_path = OUT / "klt_refresh_point_counts.csv"
variant_result["counts"].to_csv(counts_path, index=False)

print("Saved:", out_npz)
print("Saved:", counts_path)
print({
    "elapsed_s": variant_result["elapsed_s"],
    "mean_persistent_tracked_points": float(variant_result["counts"]["persistent_tracked_points"].mean()),
    "mean_persistent_points_after_refresh": float(variant_result["counts"]["persistent_points_after_refresh"].mean()),
    "mean_redetect_points": float(variant_result["counts"]["redetect_points"].mean()),
    "persistent_affine_rate": float(variant_result["counts"]["persistent_affine_ok"].mean()),
    "redetect_affine_rate": float(variant_result["counts"]["redetect_affine_ok"].mean()),
    "persistent_reinit_count": int(variant_result["counts"]["persistent_reinitialized"].sum()),
})

processed 501/2666 | persistent_pts=258 redetect_pts=300 elapsed=3.8s


processed 1001/2666 | persistent_pts=256 redetect_pts=300 elapsed=7.3s


processed 1501/2666 | persistent_pts=254 redetect_pts=300 elapsed=10.9s


processed 2001/2666 | persistent_pts=254 redetect_pts=300 elapsed=14.8s


processed 2501/2666 | persistent_pts=254 redetect_pts=300 elapsed=18.4s


processed 2666/2666 | persistent_pts=246 redetect_pts=300 elapsed=19.6s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_variant_arrays.npz
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_point_counts.csv
{'elapsed_s': 19.572914361953735, 'mean_persistent_tracked_points': 257.15722326454033, 'mean_persistent_points_after_refresh': 257.140712945591, 'mean_redetect_points': 298.84427767354595, 'persistent_affine_rate': 1.0, 'redetect_affine_rate': 1.0, 'persistent_reinit_count': 0}


## Variant metrics

The same target thresholds from the previous gates are used here: angle RMSE under 1 degree, length RMSE under 2 mm, and endpoint RMSE under 2 px.

In [6]:
def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or row["rmse"] <= target_rmse)
    return row


def metrics_for_variant(name: str, estimate: np.ndarray) -> list[dict]:
    valid = np.arange(mat_n) > 0
    ref = mat_fascicle[valid]
    est = np.asarray(estimate, dtype=float)[valid]
    length_target_px = 2.0 / mm_per_px
    return [
        metric(f"{name}_x_sup_px", ref[:, 0], est[:, 0], "px", target_rmse=2.0),
        metric(f"{name}_y_sup_px", ref[:, 1], est[:, 1], "px", target_rmse=2.0),
        metric(f"{name}_x_deep_px", ref[:, 2], est[:, 2], "px", target_rmse=2.0),
        metric(f"{name}_y_deep_px", ref[:, 3], est[:, 3], "px", target_rmse=2.0),
        metric(f"{name}_angle_deg", normalized_angles_deg(ref), normalized_angles_deg(est), "deg", target_rmse=1.0),
        metric(f"{name}_length_px", line_lengths_batch(ref), line_lengths_batch(est), "px", target_rmse=length_target_px),
        metric(f"{name}_length_mm", line_lengths_batch(ref) * mm_per_px, line_lengths_batch(est) * mm_per_px, "mm", target_rmse=2.0),
    ]


metric_rows = []
for variant_name, estimate in variant_result["outputs"].items():
    rows = metrics_for_variant(variant_name, estimate)
    for row in rows:
        row["variant"] = variant_name
    metric_rows.extend(rows)

metrics_df = pd.DataFrame(metric_rows)
metrics_df = metrics_df[["variant", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
metrics_path = OUT / "klt_refresh_variant_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
display(metrics_df)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_variant_metrics.csv


,variant,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,persistent_cumulative,persistent_cumulative_x_sup_px,px,2665,8.033085,13.460461,15.812635,0.089241,2.000000,False
1,persistent_cumulative,persistent_cumulative_y_sup_px,px,2665,6.647130,6.693206,7.197615,0.546501,2.000000,False
2,persistent_cumulative,persistent_cumulative_x_deep_px,px,2665,-95.947763,96.550538,127.003392,0.656037,2.000000,False
3,persistent_cumulative,persistent_cumulative_y_deep_px,px,2665,-9.810202,10.121265,11.497771,-0.095016,2.000000,False
4,persistent_cumulative,persistent_cumulative_angle_deg,deg,2665,-5.072719,5.073426,6.119329,0.709488,1.000000,False
5,persistent_cumulative,persistent_cumulative_length_px,px,2665,88.448185,94.174154,121.038391,0.643025,22.169625,False
6,persistent_cumulative,persistent_cumulative_length_mm,mm,2665,7.979222,8.495782,10.919300,0.643025,2.000000,False
7,persistent_one_step,persistent_one_step_x_sup_px,px,2665,-0.001159,0.648231,1.220576,0.995126,2.000000,True
8,persistent_one_step,persistent_one_step_y_sup_px,px,2665,0.001497,0.169334,0.289169,0.995046,2.000000,True
9,persistent_one_step,persistent_one_step_x_deep_px,px,2665,-0.031457,3.388943,5.802571,0.998580,2.000000,False


In [7]:
def metric_value(variant: str, suffix: str, col: str = "rmse") -> float:
    comparison = f"{variant}_{suffix}"
    return float(metrics_df.loc[metrics_df["comparison"] == comparison, col].iloc[0])

summary_rows = []
for variant in variant_result["outputs"]:
    summary_rows.append({
        "variant": variant,
        "point_policy": "persistent" if variant.startswith("persistent") else "redetect_every_step",
        "geometry_policy": "cumulative_python_geometry" if variant.endswith("cumulative") else "one_step_matlab_previous_geometry",
        "angle_rmse_deg": metric_value(variant, "angle_deg"),
        "length_rmse_px": metric_value(variant, "length_px"),
        "length_rmse_mm": metric_value(variant, "length_mm"),
        "x_sup_rmse_px": metric_value(variant, "x_sup_px"),
        "x_deep_rmse_px": metric_value(variant, "x_deep_px"),
        "passes_angle_length": bool(
            metric_value(variant, "angle_deg") <= 1.0
            and metric_value(variant, "length_mm") <= 2.0
        ),
        "passes_all_endpoint_strict": bool(metrics_df.loc[metrics_df["variant"] == variant, "passes"].all()),
    })

variant_summary = pd.DataFrame(summary_rows)
variant_summary_path = OUT / "klt_refresh_variant_summary.csv"
variant_summary.to_csv(variant_summary_path, index=False)
print("Saved:", variant_summary_path)
display(variant_summary)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_variant_summary.csv


,variant,point_policy,geometry_policy,angle_rmse_deg,length_rmse_px,length_rmse_mm,x_sup_rmse_px,x_deep_rmse_px,passes_angle_length,passes_all_endpoint_strict
0,persistent_cumulative,persistent,cumulative_python_geometry,6.119329,121.038391,10.919300,15.812635,127.003392,False,False
1,persistent_one_step,persistent,one_step_matlab_previous_geometry,0.305148,5.240469,0.472761,1.220576,5.802571,True,False
2,redetect_cumulative,redetect_every_step,cumulative_python_geometry,6.067665,130.410502,11.764791,15.006497,136.330913,False,False
3,redetect_one_step,redetect_every_step,one_step_matlab_previous_geometry,0.301311,5.167994,0.466223,1.242615,5.725075,True,False


## Affine scale diagnostics

A very small per-frame scale bias can become large when applied cumulatively. This table compares the scale implied by each Python affine against the scale of MATLAB's raw KLT segment from `f-1` to `f`.

In [8]:
def segment_scale_from_affine(prev_segment: np.ndarray, affine: np.ndarray) -> float:
    if not np.all(np.isfinite(affine)):
        return np.nan
    transformed = apply_affine_1b(prev_segment, affine)
    denom = line_lengths_batch(np.asarray(prev_segment, dtype=float).reshape(1, 4))[0]
    if not np.isfinite(denom) or denom <= 0:
        return np.nan
    return float(line_lengths_batch(transformed.reshape(1, 4))[0] / denom)


def segment_angle_delta_from_affine(prev_segment: np.ndarray, affine: np.ndarray) -> float:
    if not np.all(np.isfinite(affine)):
        return np.nan
    transformed = apply_affine_1b(prev_segment, affine)
    prev_angle = normalized_angles_deg(prev_segment.reshape(1, 4))[0]
    new_angle = normalized_angles_deg(transformed.reshape(1, 4))[0]
    return float(angle_delta_deg(new_angle, prev_angle))


scale_rows = []
for frame in range(1, mat_n):
    prev_seg = mat_fascicle[frame - 1]
    ref_scale = mat_length_px[frame] / mat_length_px[frame - 1]
    ref_angle_delta = angle_delta_deg(mat_angle[frame], mat_angle[frame - 1])
    row = {
        "frame": frame,
        "reference_segment_scale": ref_scale,
        "reference_angle_delta_deg": ref_angle_delta,
    }
    for prefix, matrices in [
        ("persistent", variant_result["persistent_affine"]),
        ("redetect", variant_result["redetect_affine"]),
    ]:
        affine = matrices[frame]
        linear = affine[:, :2] if np.all(np.isfinite(affine)) else np.full((2, 2), np.nan)
        singular = np.linalg.svd(linear, compute_uv=False) if np.all(np.isfinite(linear)) else [np.nan, np.nan]
        det = float(np.linalg.det(linear)) if np.all(np.isfinite(linear)) else np.nan
        seg_scale = segment_scale_from_affine(prev_seg, affine)
        angle_delta = segment_angle_delta_from_affine(prev_seg, affine)
        row[f"{prefix}_segment_scale"] = seg_scale
        row[f"{prefix}_scale_ratio_to_reference"] = seg_scale / ref_scale if np.isfinite(seg_scale) and ref_scale != 0 else np.nan
        row[f"{prefix}_angle_delta_deg"] = angle_delta
        row[f"{prefix}_angle_delta_error_deg"] = angle_delta - ref_angle_delta if np.isfinite(angle_delta) else np.nan
        row[f"{prefix}_linear_det"] = det
        row[f"{prefix}_singular_min"] = float(np.min(singular))
        row[f"{prefix}_singular_max"] = float(np.max(singular))
    scale_rows.append(row)

affine_diag = pd.DataFrame(scale_rows)
affine_diag_path = OUT / "klt_refresh_affine_scale_diagnostics.csv"
affine_diag.to_csv(affine_diag_path, index=False)
print("Saved:", affine_diag_path)

scale_summary_rows = []
for prefix in ["persistent", "redetect"]:
    ratio = affine_diag[f"{prefix}_scale_ratio_to_reference"].to_numpy(dtype=float)
    angle_err = affine_diag[f"{prefix}_angle_delta_error_deg"].to_numpy(dtype=float)
    finite = np.isfinite(ratio) & (ratio > 0)
    scale_summary_rows.append({
        "affine_stream": prefix,
        "n": int(finite.sum()),
        "mean_scale_ratio_minus_1": float(np.nanmean(ratio - 1.0)),
        "median_scale_ratio_minus_1": float(np.nanmedian(ratio - 1.0)),
        "rmse_scale_ratio_minus_1": float(np.sqrt(np.nanmean((ratio - 1.0) ** 2))),
        "cumulative_log_scale_ratio": float(np.nansum(np.log(ratio[finite]))),
        "exp_cumulative_log_scale_ratio": float(np.exp(np.nansum(np.log(ratio[finite])))),
        "angle_delta_error_rmse_deg": float(np.sqrt(np.nanmean(angle_err ** 2))),
    })

scale_summary = pd.DataFrame(scale_summary_rows)
scale_summary_path = OUT / "klt_refresh_affine_scale_summary.csv"
scale_summary.to_csv(scale_summary_path, index=False)
print("Saved:", scale_summary_path)
display(scale_summary)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_affine_scale_diagnostics.csv
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_affine_scale_summary.csv


,affine_stream,n,mean_scale_ratio_minus_1,median_scale_ratio_minus_1,rmse_scale_ratio_minus_1,cumulative_log_scale_ratio,exp_cumulative_log_scale_ratio,angle_delta_error_rmse_deg
0,persistent,2665,0.000053,-0.000051,0.009156,0.031713,1.032221,0.305148
1,redetect,2665,0.000041,-0.000053,0.009017,0.001148,1.001149,0.301311


## Threshold onset by variant

This shows when each variant first exceeds practical error thresholds. The cumulative variants drift early; the one-step variants stay mostly within the angle/length gates and only fail the strict endpoint gate on a small subset of transitions.

In [9]:
per_frame_rows = []
for variant, est in variant_result["outputs"].items():
    est = np.asarray(est, dtype=float)
    angle_error = angle_delta_deg(normalized_angles_deg(est), mat_angle)
    length_error_px = line_lengths_batch(est) - mat_length_px
    endpoint_abs = np.nanmax(np.abs(est - mat_fascicle), axis=1)
    for frame in range(1, mat_n):
        per_frame_rows.append({
            "variant": variant,
            "frame": frame,
            "angle_error_deg": angle_error[frame],
            "length_error_px": length_error_px[frame],
            "length_error_mm": length_error_px[frame] * mm_per_px,
            "max_endpoint_abs_error_px": endpoint_abs[frame],
        })
per_frame = pd.DataFrame(per_frame_rows)
per_frame_path = OUT / "klt_refresh_variant_per_frame_errors.csv"
per_frame.to_csv(per_frame_path, index=False)

threshold_rows = []
for variant, group in per_frame.groupby("variant"):
    for condition, mask in [
        ("abs_angle_gt_1deg", group["angle_error_deg"].abs() > 1.0),
        ("abs_length_gt_2mm", group["length_error_mm"].abs() > 2.0),
        ("endpoint_gt_10px", group["max_endpoint_abs_error_px"] > 10.0),
    ]:
        frames = group.loc[mask, "frame"].to_numpy(dtype=int)
        threshold_rows.append({
            "variant": variant,
            "condition": condition,
            "first_frame": int(frames[0]) if len(frames) else None,
            "n_frames": int(len(frames)),
            "fraction_frames": float(len(frames) / (mat_n - 1)),
        })
threshold_df = pd.DataFrame(threshold_rows)
threshold_path = OUT / "klt_refresh_threshold_onset.csv"
threshold_df.to_csv(threshold_path, index=False)

print("Saved:", per_frame_path)
print("Saved:", threshold_path)
display(threshold_df)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_variant_per_frame_errors.csv
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_threshold_onset.csv


,variant,condition,first_frame,n_frames,fraction_frames
0,persistent_cumulative,abs_angle_gt_1deg,43,2346,0.880300
1,persistent_cumulative,abs_length_gt_2mm,42,1914,0.718199
2,persistent_cumulative,endpoint_gt_10px,41,2549,0.956473
3,persistent_one_step,abs_angle_gt_1deg,102,58,0.021764
4,persistent_one_step,abs_length_gt_2mm,157,19,0.007129
5,persistent_one_step,endpoint_gt_10px,102,179,0.067167
6,redetect_cumulative,abs_angle_gt_1deg,43,2429,0.911445
7,redetect_cumulative,abs_length_gt_2mm,42,1943,0.729081
8,redetect_cumulative,endpoint_gt_10px,41,2498,0.937336
9,redetect_one_step,abs_angle_gt_1deg,102,54,0.020263


## Decision

This notebook should answer whether to spend the next effort on point-refresh policy alone. If both cumulative variants drift similarly while both one-step variants remain close, then changing the point refresh threshold is not enough. We need to correct how small affine scale/endpoint errors are accumulated, or introduce the MATLAB state-estimation correction at the right boundary.

In [10]:
persistent_cum_len = metric_value("persistent_cumulative", "length_mm")
redetect_cum_len = metric_value("redetect_cumulative", "length_mm")
persistent_step_len = metric_value("persistent_one_step", "length_mm")
redetect_step_len = metric_value("redetect_one_step", "length_mm")

summary = {
    "gate": "UltraTrack/KLT sequential refresh variants",
    "frames": int(mat_n),
    "persistent_cumulative_length_rmse_mm": persistent_cum_len,
    "redetect_cumulative_length_rmse_mm": redetect_cum_len,
    "persistent_one_step_length_rmse_mm": persistent_step_len,
    "redetect_one_step_length_rmse_mm": redetect_step_len,
    "persistent_cumulative_angle_rmse_deg": metric_value("persistent_cumulative", "angle_deg"),
    "redetect_cumulative_angle_rmse_deg": metric_value("redetect_cumulative", "angle_deg"),
    "persistent_one_step_angle_rmse_deg": metric_value("persistent_one_step", "angle_deg"),
    "redetect_one_step_angle_rmse_deg": metric_value("redetect_one_step", "angle_deg"),
    "persistent_reinit_count": int(variant_result["counts"]["persistent_reinitialized"].sum()),
    "mean_persistent_points_after_refresh": float(variant_result["counts"]["persistent_points_after_refresh"].mean()),
    "mean_redetect_points": float(variant_result["counts"]["redetect_points"].mean()),
    "conclusion": "Point refresh alone does not fix sequential drift; one-step affines are close, but small affine/endpoint errors compound when integrated cumulatively.",
    "next_action": "Notebook 48 should inspect cumulative affine scale correction and/or implement MATLAB 2-state Kalman using raw KLT predictions plus TimTrack measurements.",
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

readiness = pd.DataFrame([
    {
        "gate": "TimTrack mask/doHough",
        "status": "passed",
        "ready_for_next": True,
    },
    {
        "gate": "KLT local one-step affine",
        "status": "near-pass and stable across persistent/redetect point policies",
        "ready_for_next": True,
    },
    {
        "gate": "KLT sequential raw parity",
        "status": "not passed; cumulative affine integration drifts even with re-detect every step",
        "ready_for_next": False,
    },
    {
        "gate": "MATLAB 2-state Kalman",
        "status": "next downstream correction candidate, but keep separate from KLT raw gate",
        "ready_for_next": True,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
print("Saved:", readiness_path)
display(readiness)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/summary.json
{
  "gate": "UltraTrack/KLT sequential refresh variants",
  "frames": 2666,
  "persistent_cumulative_length_rmse_mm": 10.919299696433825,
  "redetect_cumulative_length_rmse_mm": 11.764790872578969,
  "persistent_one_step_length_rmse_mm": 0.4727611533916675,
  "redetect_one_step_length_rmse_mm": 0.46622294045826423,
  "persistent_cumulative_angle_rmse_deg": 6.11932925350646,
  "redetect_cumulative_angle_rmse_deg": 6.067665384511656,
  "persistent_one_step_angle_rmse_deg": 0.3051483694158685,
  "redetect_one_step_angle_rmse_deg": 0.3013109932225762,
  "persistent_reinit_count": 0,
  "mean_persistent_points_after_refresh": 257.140712945591,
  "mean_redetect_points": 298.84427767354595,
  "conclusion": "Point refresh alone does not fix sequential drift; one-step affines are close, but small affine/endpoint errors compound when integrated cumulatively.",
  "next_action":

,gate,status,ready_for_next
0,TimTrack mask/doHough,passed,True
1,KLT local one-step affine,near-pass and stable across persistent/redetec...,True
2,KLT sequential raw parity,not passed; cumulative affine integration drif...,False
3,MATLAB 2-state Kalman,"next downstream correction candidate, but keep...",True
